# Agentic Full vs Baseline (HPC)

This notebook runs `agentic_full` and `baseline` in separate cells, then compares results and inspects agentic artifacts.


In [ ]:
import os
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpc_llm_setup import (
    config_from_env,
    ensure_eoh_src_on_path,
    start_hpc_bridge,
    stop_hpc_bridge,
    test_bridge,
)

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
PROJECT_ROOT, EOH_SRC = ensure_eoh_src_on_path()
COMPARE_ROOT = Path(os.getenv('EOH_COMPARE_OUT', './compare_runs')).resolve()
COMPARE_ROOT.mkdir(parents=True, exist_ok=True)

SEED = int(os.getenv('EOH_SEED', '2024'))
SEED_ROOT = COMPARE_ROOT / f'seed_{SEED}'
SEED_ROOT.mkdir(parents=True, exist_ok=True)

AGENTIC_GENS = int(os.getenv('EOH_AGENTIC_GENS', '40'))
BASELINE_GENS = int(os.getenv('EOH_BASELINE_GENS', '10'))

SETTINGS = {
    'problem': os.getenv('EOH_PROBLEM', 'bp_online'),
    'pop_size': int(os.getenv('EOH_POP_SIZE', '8')),
    'n_proc': int(os.getenv('EOH_N_PROC', str(min(24, os.cpu_count() or 1)))),
    'eval_parallel_instances': int(os.getenv('EOH_EVAL_PARALLEL_INSTANCES', '1')),
    'eva_timeout': int(os.getenv('EOH_EVA_TIMEOUT', '240')),
    'eval_instances_per_gen': int(os.getenv('EOH_EVAL_INSTANCES_PER_GEN', '256')),
    'holdout_instances': int(os.getenv('EOH_HOLDOUT_INSTANCES', '64')),
    'holdout_eval_interval': int(os.getenv('EOH_HOLDOUT_EVAL_INTERVAL', '1')),
    'route_improvement_epsilon': float(os.getenv('EOH_ROUTE_IMPROVEMENT_EPS', '1e-12')),
    'route_warmup_gens': int(os.getenv('EOH_ROUTE_WARMUP_GENS', '2')),
    'route_e1_cooldown': int(os.getenv('EOH_ROUTE_E1_COOLDOWN', '3')),
    'route_e2_recent_k': int(os.getenv('EOH_ROUTE_E2_RECENT_K', '3')),
    'route_use_probabilistic': os.getenv('EOH_ROUTE_USE_PROBABILISTIC', '1') == '1',
    'route_shuffle_operator_order': os.getenv('EOH_ROUTE_SHUFFLE_OPERATOR_ORDER', '0') == '1',
    'route_controller_enabled': os.getenv('EOH_ROUTE_CONTROLLER_ENABLED', '1') == '1',
    'route_controller_window': int(os.getenv('EOH_ROUTE_CONTROLLER_WINDOW', '5')),
    'route_controller_use_critic': os.getenv('EOH_ROUTE_CONTROLLER_USE_CRITIC', '0') == '1',
    'disable_numba': os.getenv('EOH_DISABLE_NUMBA', '1') == '1',
}

os.environ.setdefault('EOH_LOCAL_LLM_TIMEOUT_S', '600')
os.environ.setdefault('EOH_OFFSPRING_RETRIES', '2')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('COMPARE_ROOT:', COMPARE_ROOT)
print('SEED_ROOT:', SEED_ROOT)
print('AGENTIC_GENS:', AGENTIC_GENS)
print('BASELINE_GENS:', BASELINE_GENS)
print('SETTINGS:', SETTINGS)


In [ ]:
from eoh import eoh
from eoh.utils.getParas import Paras


def _build_shared_seed_if_needed(seed_root: Path, bridge_url: str, model_id: str, settings: dict):
    shared_seed_path = seed_root / 'shared_initial_population.json'
    if shared_seed_path.exists():
        print(f'[shared-seed] using existing: {shared_seed_path}')
        return shared_seed_path

    seed_build_out = seed_root / '_shared_seed_build'
    if seed_build_out.exists():
        shutil.rmtree(seed_build_out)
    seed_build_out.mkdir(parents=True, exist_ok=True)

    random.seed(SEED)
    np.random.seed(SEED)

    paras = Paras()
    paras.set_paras(
        method='eoh',
        problem=settings['problem'],
        llm_use_local=True,
        llm_local_url=bridge_url,
        llm_model=model_id,
        ec_pop_size=settings['pop_size'],
        ec_n_pop=0,
        exp_n_proc=settings['n_proc'],
        exp_output_path=str(seed_build_out),
        exp_debug_mode=False,
        eva_timeout=settings['eva_timeout'],
        eval_parallel_instances=settings['eval_parallel_instances'],
        eval_instances_per_gen=settings['eval_instances_per_gen'],
        holdout_instances=settings['holdout_instances'],
        holdout_eval_interval=settings['holdout_eval_interval'],
        route_improvement_epsilon=settings['route_improvement_epsilon'],
        route_warmup_gens=settings['route_warmup_gens'],
        route_e1_cooldown=settings['route_e1_cooldown'],
        route_e2_recent_k=settings['route_e2_recent_k'],
        route_use_probabilistic=settings['route_use_probabilistic'],
        route_controller_enabled=settings['route_controller_enabled'],
        route_controller_window=settings['route_controller_window'],
        eoh_mode='baseline',
        log_full_population=True,
    )
    if settings.get('disable_numba', False):
        paras.eva_numba_decorator = False

    runner = eoh.EVOL(paras)
    runner.run()

    pop0_path = seed_build_out / 'results' / 'pops' / 'population_generation_0.json'
    if not pop0_path.exists():
        raise RuntimeError(f'missing {pop0_path}')

    with pop0_path.open('r', encoding='utf-8') as f:
        pop0 = json.load(f)

    seeds = []
    for ind in pop0:
        if not isinstance(ind, dict):
            continue
        code = ind.get('code')
        algorithm = ind.get('algorithm')
        if isinstance(code, str) and isinstance(algorithm, str):
            seeds.append({'algorithm': algorithm, 'code': code})

    if len(seeds) == 0:
        raise RuntimeError('no valid seed algorithms built')

    with shared_seed_path.open('w', encoding='utf-8') as f:
        json.dump(seeds, f, indent=2)

    print(f'[shared-seed] created: {shared_seed_path} ({len(seeds)} seeds)')
    return shared_seed_path


def _run_single_mode(mode: str, generations: int, mode_out: Path, bridge_url: str, model_id: str, settings: dict, shared_seed_path: Path, use_critic: bool | None = None):
    if mode_out.exists():
        shutil.rmtree(mode_out)
    mode_out.mkdir(parents=True, exist_ok=True)

    random.seed(SEED)
    np.random.seed(SEED)

    paras = Paras()
    if use_critic is None:
        use_critic = bool(settings.get('route_controller_use_critic', False))

    paras.set_paras(
        method='eoh',
        problem=settings['problem'],
        llm_use_local=True,
        llm_local_url=bridge_url,
        llm_model=model_id,
        ec_pop_size=settings['pop_size'],
        ec_n_pop=int(generations),
        exp_n_proc=settings['n_proc'],
        exp_output_path=str(mode_out),
        exp_debug_mode=False,
        eva_timeout=settings['eva_timeout'],
        eval_parallel_instances=settings['eval_parallel_instances'],
        eval_instances_per_gen=settings['eval_instances_per_gen'],
        holdout_instances=settings['holdout_instances'],
        holdout_eval_interval=settings['holdout_eval_interval'],
        route_improvement_epsilon=settings['route_improvement_epsilon'],
        route_warmup_gens=settings['route_warmup_gens'],
        route_e1_cooldown=settings['route_e1_cooldown'],
        route_e2_recent_k=settings['route_e2_recent_k'],
        route_use_probabilistic=settings['route_use_probabilistic'],
        route_shuffle_operator_order=bool(settings.get('route_shuffle_operator_order', False)),
        route_controller_enabled=settings['route_controller_enabled'],
        route_controller_window=settings['route_controller_window'],
        route_controller_use_critic=bool(use_critic),
        exp_use_seed=True,
        exp_seed_path=str(shared_seed_path),
        eoh_mode=mode,
        log_full_population=False,
    )
    if settings.get('disable_numba', False):
        paras.eva_numba_decorator = False

    print(f'[run] mode={mode} gens={generations} use_critic={bool(use_critic)} out={mode_out}')
    runner = eoh.EVOL(paras)
    runner.run()
    print(f'[run] done mode={mode}')


def run_mode_with_bridge(mode: str, generations: int, output_name: str | None = None, use_critic: bool | None = None):
    cfg = config_from_env()
    server = None
    try:
        server, _thread, bridge_url, model_id = start_hpc_bridge(cfg)
        status, payload = test_bridge(bridge_url)
        print('[bridge] status:', status, '| payload:', payload)

        shared_seed_path = _build_shared_seed_if_needed(SEED_ROOT, bridge_url, model_id, SETTINGS)
        mode_name = output_name or mode
        mode_out = SEED_ROOT / mode_name
        _run_single_mode(mode, generations, mode_out, bridge_url, model_id, SETTINGS, shared_seed_path, use_critic=use_critic)
    finally:
        stop_hpc_bridge(server)
        print('[bridge] stopped')


## Run 1: Agentic Full
Run this cell first. If interrupted, rerun only this cell.


In [ ]:
run_mode_with_bridge('agentic_full', AGENTIC_GENS, output_name='agentic_full', use_critic=False)


## Run 2: Baseline
Run this after agentic_full. If interrupted, rerun only this cell.


In [ ]:
run_mode_with_bridge('baseline', BASELINE_GENS, output_name='baseline', use_critic=False)


In [ ]:
def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows


def load_mode_df(seed_root: Path, mode: str):
    p = seed_root / mode / 'results' / 'run_log.jsonl'
    return pd.DataFrame(read_jsonl(p))


baseline_df = load_mode_df(SEED_ROOT, 'baseline')
agentic_df = load_mode_df(SEED_ROOT, 'agentic_full')

print('baseline rows:', len(baseline_df))
print('agentic_full rows:', len(agentic_df))


def _last_best(df):
    if len(df) == 0 or 'best_fitness' not in df.columns:
        return None
    return float(df['best_fitness'].iloc[-1])

print('final best baseline:', _last_best(baseline_df))
print('final best agentic_full:', _last_best(agentic_df))

if len(agentic_df):
    display(agentic_df.tail(5))
if len(baseline_df):
    display(baseline_df.tail(5))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

if not baseline_df.empty:
    yb = baseline_df['train_fitness'] if 'train_fitness' in baseline_df.columns else baseline_df['best_fitness']
    axes[0].plot(baseline_df['gen'], yb, label='baseline')
if not agentic_df.empty:
    ya = agentic_df['train_fitness'] if 'train_fitness' in agentic_df.columns else agentic_df['best_fitness']
    axes[0].plot(agentic_df['gen'], ya, label='agentic_full')
axes[0].set_title('Train Fitness vs Generation')
axes[0].set_xlabel('gen')
axes[0].set_ylabel('fitness (lower better)')
axes[0].legend()

if not baseline_df.empty:
    axes[1].plot(baseline_df['gen'], baseline_df['invalid_rate'], label='baseline')
if not agentic_df.empty:
    axes[1].plot(agentic_df['gen'], agentic_df['invalid_rate'], label='agentic_full')
axes[1].set_title('Invalid Rate vs Generation')
axes[1].set_xlabel('gen')
axes[1].set_ylabel('invalid_rate')
axes[1].legend()

if not agentic_df.empty and 'chosen_operator' in agentic_df.columns:
    op_counts = agentic_df['chosen_operator'].astype(str).value_counts()
    axes[2].bar(op_counts.index, op_counts.values)
axes[2].set_title('Agentic Full Operator Counts')
axes[2].set_xlabel('operator')
axes[2].set_ylabel('count')

plt.tight_layout()
plt.show()


In [ ]:
artifact_files = [
    'measurement_plan.jsonl',
    'behavior_evidence.jsonl',
    'diagnosis_report.jsonl',
    'intervention_portfolio.jsonl',
    'reflection_report.jsonl',
    'heuristic_profiles.jsonl',
    'memory_updates.jsonl',
    'agent_observation.jsonl',
    'agent_diagnosis.jsonl',
    'agent_plan.jsonl',
    'agent_critic.jsonl',
]

for fname in artifact_files:
    p = SEED_ROOT / 'agentic_full' / 'results' / fname
    rows = read_jsonl(p)
    print(fname, 'rows=', len(rows), 'path=', p)
    if rows:
        display(pd.DataFrame(rows).tail(2))
